# All Convolutional Net Regularization With Transfer Learning


In [4]:
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics

class CIFAR10ConvNet(pl.LightningModule):
    def __init__(self, class_total=10, dropout_strength=0.5):
        super(CIFAR10ConvNet, self).__init__()
        # Layer setup with increased filter counts and dropout for regularization
        self.first_conv = nn.Conv2d(1, 96, kernel_size=3, padding=1)
        self.second_conv = nn.Conv2d(96, 96, kernel_size=3, padding=1)
        self.first_dropout = nn.Dropout(dropout_strength)
        self.third_conv = nn.Conv2d(96, 96, kernel_size=3, stride=2, padding=1)  # Downsampling with stride
        self.fourth_conv = nn.Conv2d(96, 192, kernel_size=3, padding=1)
        self.fifth_conv = nn.Conv2d(192, 192, kernel_size=3, padding=1)
        self.second_dropout = nn.Dropout(dropout_strength)
        self.sixth_conv = nn.Conv2d(192, 192, kernel_size=3, stride=2, padding=1)  # Further reduce spatial dimensions
        self.seventh_conv = nn.Conv2d(192, 192, kernel_size=3, padding=1)
        self.eighth_conv = nn.Conv2d(192, 192, kernel_size=1)  # 1x1 convolution for dimensionality reduction
        self.ninth_conv = nn.Conv2d(192, class_total, kernel_size=1)  # Final convolution for class output

        self.adaptive_pool = nn.AdaptiveAvgPool2d(1)  # Global pooling to reduce overfitting
        self.performance_metrics = torchmetrics.Accuracy(num_classes=class_total, average="macro", task="multiclass")

    def forward(self, inputs):
        inputs = F.relu(self.first_conv(inputs))
        inputs = F.relu(self.second_conv(inputs))
        inputs = self.first_dropout(inputs)  # Applying dropout to prevent overfitting
        inputs = F.relu(self.third_conv(inputs))
        inputs = F.relu(self.fourth_conv(inputs))
        inputs = F.relu(self.fifth_conv(inputs))
        inputs = self.second_dropout(inputs)  # Additional dropout layer for robustness
        inputs = F.relu(self.sixth_conv(inputs))
        inputs = F.relu(self.seventh_conv(inputs))
        inputs = F.relu(self.eighth_conv(inputs))
        inputs = self.ninth_conv(inputs)
        inputs = self.adaptive_pool(inputs)
        inputs = inputs.view(inputs.size(0), -1)  # Flatten the output for classification
        return inputs

    def on_train_batch(self, batch, batch_idx):
        data, labels = batch
        predictions = self(data)
        training_loss = F.cross_entropy(predictions, labels)
        self.log("training_loss", training_loss, on_step=True, on_epoch=True, prog_bar=True, logger=True)
        return training_loss

    def on_validate_batch(self, batch, batch_idx):
        data, labels = batch
        predictions = self(data)
        validation_loss = F.cross_entropy(predictions, labels)
        self.performance_metrics(predictions, labels)
        self.log("validation_loss", validation_loss, on_epoch=True, prog_bar=True)
        self.log("validation_accuracy", self.performance_metrics, on_epoch=True, prog_bar=True)

    def on_test_batch(self, batch, batch_idx):
        data, labels = batch
        predictions = self(data)
        test_loss = F.cross_entropy(predictions, labels)
        self.performance_metrics(predictions, labels)
        self.log("test_accuracy", self.performance_metrics)
        self.log("test_loss", test_loss)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=0.001)



The CIFAR10 dataset is available to download from `torchvision`, as shown in the cell below.

**Note: After downloading the CIFAR10 dataset, you will need to set `download=False` in the cell below to avoid errors.**

In [5]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from torchvision import transforms
from torchvision.datasets import CIFAR10
import torch
from torch.utils.data import DataLoader, random_split

# Enhance the training data with various augmentation techniques
enhanced_train_transforms = transforms.Compose([
    transforms.CenterCrop(160),
    transforms.Resize(64),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),  # Allow image rotation to enhance generalization
    transforms.RandomResizedCrop(256, scale=(0.8, 1.0)),  # Implement random cropping
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    transforms.Grayscale(),  # Convert images to grayscale to simplify the model
])

# Standardized transformations for evaluation to ensure consistency
standard_test_transforms = transforms.Compose([
    transforms.CenterCrop(160),
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
    transforms.Grayscale()
])

# Load CIFAR10 training data with augmentations
cifar10_train_data = CIFAR10("../data/CIFAR10/train/", train=True, download=True, transform=enhanced_train_transforms)

# Split the training data to create a validation subset
size_of_training_set = int(0.9 * len(cifar10_train_data))
size_of_validation_set = len(cifar10_train_data) - size_of_training_set

# Ensure reproducibility when splitting the dataset
dataset_seed = torch.Generator().manual_seed(42)
train_subset, validation_subset = random_split(cifar10_train_data, [size_of_training_set, size_of_validation_set], generator=dataset_seed)
validation_subset.dataset.transform = standard_test_transforms

# DataLoader configuration for managing training and validation data
training_dataloader = DataLoader(
    train_subset, batch_size=128, num_workers=8, shuffle=True, persistent_workers=True
)
validation_dataloader = DataLoader(
    validation_subset, batch_size=128, num_workers=8, shuffle=False, persistent_workers=True
)

# Prepare the test dataset with consistent preprocessing
cifar10_test_data = CIFAR10("../data/CIFAR10/test/", train=False, download=True, transform=standard_test_transforms)

# Instantiate and load a pre-trained model for fine-tuning
network_model = AllConvNetCIFAR10()
network_model.load_state_dict(torch.load("../models/all_convnet.pth"))

# Setup EarlyStopping to prevent overfitting
adaptive_early_stopping = EarlyStopping(
    monitor="val_loss",
    mode="min",
    patience=5
)

# Configuration for saving the best-performing model during training
optimized_checkpoint_saver = ModelCheckpoint(
    monitor="val_loss",
    mode="min"
)


Files already downloaded and verified
Files already downloaded and verified


/var/folders/w6/17qnjd6d5xq8m2w1crldcn1c0000gn/T/ipykernel_59568/2888254110.py:53: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  network_model.load_state_dict(torch.load("..

In [3]:
# Fit the model
torch.set_float32_matmul_precision("high")
trainer = L.Trainer(callbacks=[early_stop_callback, checkpoint_callback], max_epochs=-1)

trainer.fit(model=model, train_dataloaders=train_loader, val_dataloaders=val_loader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

   | Name            | Type               | Params | Mode 
----------------------------------------------------------------
0  | conv1           | Conv2d             | 960    | train
1  | conv2           | Conv2d             | 83.0 K | train
2  | dropout1        | Dropout            | 0      | train
3  | conv3         

Sanity Checking: |                                        | 0/? [00:00<?, ?it/s]

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/spawn.py", line 116, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/multiprocessing/spawn.py", line 126, in _main
    self = reduction.pickle.load(from_parent)
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/__init__.py", line 10, in <module>

Detected KeyboardInterrupt, attempting graceful shutdown ...
    from torchvision import _meta_registrations, datasets, io, models, ops, transforms, utils  # usort:skip
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/models/__init__.py", line 17, in <module>
    from . import detection, optical_flow, quantization, segmentation, video
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python

NameError: name 'exit' is not defined

In [ ]:
# Evaluate the model on the test set
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=256, num_workers=8, shuffle=False, persistent_workers=True
)
trainer.test(model=model, dataloaders=test_loader)